# Preprocessing:

first we need to do

    - Outlier Detection
    - Train Test Split

## Import the libraries



In [1]:
import sys, os
import polars as pl
import pandas as pd
import numpy as np
# from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder





sys.path.append(os.path.join(os.getcwd(), "Utils"))
BASE_DIR = os.path.abspath("..")
sys.path.append(os.path.join(BASE_DIR, "utils"))


from StatMelt import iqr_bounds

## Outlier Detection:
- in the EDA section We found out that there is no outliers in my dataset
- there is no Missing values in this dataset.

In [2]:
df = pd.read_csv('../Data/Left/LeftFinal.csv')
df.head()

,Unnamed: 0,day,date,time_of_day,Diastolic,Systolic,Pulse
0,0,3.0,1405/2/10,morning,83.0,127.0,64.0
1,1,3.0,1405/2/10,noon,66.0,113.0,67.0
2,2,3.0,1405/2/10,night,77.0,138.0,69.0
3,3,4.0,1405/2/11,morning,75.0,128.0,54.0
4,4,4.0,1405/2/11,noon,74.0,141.0,72.0


In [3]:
iqr_bounds(df['Diastolic'])

(np.float64(62.5), np.float64(82.5), np.float64(5.0))

## Train Test Split
- Because this is a time series dataset and I want to do Time series Analysis on this data then I should just get the values from the last 20% of the dataset as the test set.

- I need to drop the columns that are not useful.
    - the first column is `day` because it will surve no purpose to be in My Time series Analysis.
    - the second column is `Pulse` because from EDA I found out that it has near Zero correlation to Diastolic and Systolic
    - I am not sure about what should I set as the target variable, My gut feeling says Diastolic, I will change it to Systolic and find how that variable response.

In [4]:
df

,Unnamed: 0,day,date,time_of_day,Diastolic,Systolic,Pulse
0,0,3.0,1405/2/10,morning,83.0,127.0,64.0
1,1,3.0,1405/2/10,noon,66.0,113.0,67.0
2,2,3.0,1405/2/10,night,77.0,138.0,69.0
3,3,4.0,1405/2/11,morning,75.0,128.0,54.0
4,4,4.0,1405/2/11,noon,74.0,141.0,72.0
5,5,4.0,1405/2/11,night,85.0,139.0,74.0
6,6,5.0,1405/2/12,morning,74.0,143.0,60.0
7,7,5.0,1405/2/12,noon,66.0,125.0,65.0
8,8,5.0,1405/2/12,night,74.0,131.0,68.0
9,9,6.0,1405/2/13,morning,68.0,123.0,64.0


In [5]:
df = df.drop(columns=['Unnamed: 0','day','Pulse'])
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date')
df.reset_index(drop=True)

,date,time_of_day,Diastolic,Systolic
0,1405-02-08,night,69.0,138.0
1,1405-02-08,noon,74.0,140.0
2,1405-02-08,morning,89.0,149.0
3,1405-02-09,night,75.0,137.0
4,1405-02-09,morning,78.0,151.0
5,1405-02-09,noon,75.0,141.0
6,1405-02-10,morning,83.0,127.0
7,1405-02-10,noon,66.0,113.0
8,1405-02-10,night,77.0,138.0
9,1405-02-11,night,85.0,139.0


In [27]:
train_df = df[0:24]
test_df = df[24:30]
test_df

,date,time_of_day,Diastolic,Systolic
19,1405-02-16,noon,71.0,125.0
20,1405-02-16,night,84.0,122.0
18,1405-02-16,morning,70.0,123.0
21,1405-02-17,morning,73.0,133.0
22,1405-02-17,noon,63.0,122.0
23,1405-02-17,night,70.0,123.0


In [28]:
ohe = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')
time_ohe = ohe.fit(train_df[['time_of_day']])

train_ohe = time_ohe.transform(train_df[['time_of_day']])
test_ohe = time_ohe.transform(test_df[['time_of_day']])

In [29]:
ohe_cols = time_ohe.get_feature_names_out(['time_of_day'])
ohe_cols

array(['time_of_day_night', 'time_of_day_noon'], dtype=object)

In [30]:
train_ohe_df = pd.DataFrame(train_ohe , columns=ohe_cols, index=train_df.index)
test_ohe_df = pd.DataFrame(test_ohe , columns=ohe_cols, index=test_df.index)


In [31]:
train_df

,date,time_of_day,Diastolic,Systolic
26,1405-02-08,night,69.0,138.0
25,1405-02-08,noon,74.0,140.0
24,1405-02-08,morning,89.0,149.0
29,1405-02-09,night,75.0,137.0
27,1405-02-09,morning,78.0,151.0
28,1405-02-09,noon,75.0,141.0
0,1405-02-10,morning,83.0,127.0
1,1405-02-10,noon,66.0,113.0
2,1405-02-10,night,77.0,138.0
5,1405-02-11,night,85.0,139.0


In [32]:

X_train_enc = pd.concat([train_df, train_ohe_df], axis=1)
X_test_enc = pd.concat([test_df, test_ohe_df], axis=1)

In [33]:
X_train_enc = X_train_enc.drop(columns=['time_of_day'])
X_test_enc = X_test_enc.drop(columns=['time_of_day'])

In [34]:
X_train_enc = X_train_enc.sort_values('date')
X_train_enc = X_train_enc.set_index('date')

X_train_enc

,Diastolic,Systolic,time_of_day_night,time_of_day_noon
date,,,,
1405-02-08,69.0,138.0,1.0,0.0
1405-02-08,74.0,140.0,0.0,1.0
1405-02-08,89.0,149.0,0.0,0.0
1405-02-09,75.0,137.0,1.0,0.0
1405-02-09,78.0,151.0,0.0,0.0
1405-02-09,75.0,141.0,0.0,1.0
1405-02-10,83.0,127.0,0.0,0.0
1405-02-10,66.0,113.0,0.0,1.0
1405-02-10,77.0,138.0,1.0,0.0


In [35]:
X_test_enc = X_test_enc.sort_values('date')
X_test_enc = X_test_enc.set_index('date')

X_test_enc

,Diastolic,Systolic,time_of_day_night,time_of_day_noon
date,,,,
1405-02-16,71.0,125.0,0.0,1.0
1405-02-16,84.0,122.0,1.0,0.0
1405-02-16,70.0,123.0,0.0,0.0
1405-02-17,73.0,133.0,0.0,0.0
1405-02-17,63.0,122.0,0.0,1.0
1405-02-17,70.0,123.0,1.0,0.0
